# MAM4-JAX in AMBRS

[MAM4-JAX](https://github.com/reflective-org/MAM4-JAX) is a JAX re-implementation of the
4-mode MAM4 box model: the same fixed-structure modal scheme as the executable MAM4 that
AMBRS already supports — accumulation, Aitken, coarse and primary-carbon modes with fixed
geometric standard deviations — but as a differentiable Python library.

Like TOMAS-JAX, it runs **in process**: no binaries to build, so this notebook works
straight after `pip install -r requirements.txt`.

What follows:

1. define a 4-mode scenario (MAM4-JAX's own reference initial condition),
2. run it and watch the size distribution evolve,
3. see what each microphysical process contributes,
4. watch the modes themselves (number and diameter per mode),
5. notes on comparing against the executable MAM4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import ambrs
import ambrs.aerosol as aerosol
import ambrs.gas as gas

plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,          # recessive grid
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# a fixed, colourblind-safe order (Okabe-Ito); assigned in order, never cycled
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#E69F00"]

# MAM4's output population is a smooth 4-mode lognormal, so unlike a sectional
# model there is no native bin grid to respect; an 80-point diameter grid is fine.
DIAMETERS = np.logspace(-9, -5, 80)   # [m]


def size_distribution(output, diameters=DIAMETERS):
    """dN/dlnD [# m^-3] for an ambrs Output, on the given diameter grid [m]."""
    return np.asarray(output.compute_variable("dNdlnD", {
        "diam_grid": diameters,
        "normalize": False,
        "wetsize": False,
        "method": "hist",
    }))


def style_axes(ax, title):
    ax.set_xscale("log")
    ax.set_xlabel("dry diameter [nm]")
    ax.set_ylabel("dN/dlnD [# m$^{-3}$]")
    ax.set_title(title)


def summarise(label, output):
    population = output.particle_population
    return {
        "case": label,
        "N [# m^-3]": population.get_Ntot(),
        "dry mass [kg m^-3]": population.get_tot_dry_mass(),
    }


print("mam4-jax available:", ambrs.mam4_jax._MAM4_JAX_AVAILABLE)

## 1. Define a 4-mode scenario

MAM4 requires exactly four modes, in order: accumulation, Aitken, coarse, primary carbon.
The numbers below are MAM4-JAX's own reference initial condition (its benchmark namelist)
scaled up ~30x to a polluted-boundary-layer loading, so that a day of aging is actually
visible — at the reference clean-air concentrations, coagulation moves total number by only
~15% in 24 h. Gas concentrations are MAM4 mass mixing ratios
[kg gas / kg dry air], matching `ambrs.mam4`.

In [ ]:
so4 = aerosol.AerosolSpecies(name="SO4", molar_mass=97.071, density=1770,
                             hygroscopicity=0.507)
soa = aerosol.AerosolSpecies(name="MSA", molar_mass=96.1, density=1000,
                             hygroscopicity=0.14)   # secondary organic, MAM4 conventions
ncl = aerosol.AerosolSpecies(name="Na", molar_mass=58.44, density=1900,
                             hygroscopicity=1.16)   # sea salt
pom = aerosol.AerosolSpecies(name="OC", molar_mass=12.01, density=1000,
                             hygroscopicity=0.01)
bc = aerosol.AerosolSpecies(name="BC", molar_mass=12.01, density=1700,
                            hygroscopicity=1e-10)
so2 = gas.GasSpecies(name="SO2", molar_mass=64.07)
h2so4 = gas.GasSpecies(name="H2SO4", molar_mass=98.079)


def mode(name, species, fractions, number, gmd, gsd):
    return aerosol.AerosolModeState(
        name=name, species=tuple(species), number=number,
        geom_mean_diam=gmd, log10_geom_std_dev=np.log10(gsd),
        mass_fractions=tuple(fractions))


scenario = ambrs.Scenario(
    aerosols=(so4, soa, ncl, pom, bc),
    gases=(so2, h2so4),
    size=aerosol.AerosolModalSizeState(modes=(
        mode("accumulation",   [so4, soa, ncl], [0.3, 0.3, 0.4], 3e9,  0.11e-6, 1.8),
        mode("aitken",         [so4, soa, ncl], [0.3, 0.3, 0.4], 3e10, 0.026e-6, 1.6),
        mode("coarse",         [so4, soa, ncl], [0.3, 0.3, 0.4], 1e5,  2.0e-6, 1.8),
        mode("primary carbon", [pom, bc],       [0.5, 0.5],      6e9,  0.05e-6, 1.6),
    )),
    gas_concs=(1e-4, 1e-13),   # SO2, H2SO4 [kg/kg]; SO2 is inert in this build
    flux=0.0,
    relative_humidity=0.9,
    temperature=273.0,         # [K]
    pressure=1.0e5,            # [Pa]
    height=500.0,              # [m]
)

for m in scenario.size.modes:
    print(f"{m.name:>15}: N = {m.number:.2e} /m3, GMD = {m.geom_mean_diam*1e9:7.1f} nm, "
          f"GSD = {10**m.log10_geom_std_dev:.2f}")

## 2. Run it, and watch the distribution evolve

Coagulation, condensation and nucleation together, over 24 simulated hours.
`gas_phase_chemistry` enables MAM4-JAX's built-in H2SO4 production term, which feeds
condensation and nucleation. `run_ensemble` compiles the step
(`calcsize → wateruptake → amicphys`) once and reuses it for every input.

In [ ]:
model = ambrs.mam4_jax.AerosolModel(ambrs.AerosolProcesses(
    coagulation=True, condensation=True, nucleation=True, gas_phase_chemistry=True))

DT = 300.0                     # [s]
HOURS = [0, 1, 6, 24]
inputs = [model.create_input(scenario, dt=DT, nstep=int(h * 3600 / DT) or 1)
          for h in HOURS]
# nstep must be >= 1, so "hour 0" is a single step: effectively the initial state
outputs = model.run_ensemble(inputs)

fig, ax = plt.subplots()
for colour, hours, output in zip(PALETTE, HOURS, outputs):
    ax.plot(DIAMETERS * 1e9, size_distribution(output), lw=2,
            color=colour, label=f"{hours} h")
style_axes(ax, "Size distribution over 24 h (coag + cond + nucleation)")
ax.legend(title="elapsed", frameon=False)
plt.tight_layout()
plt.show()

pd.DataFrame([summarise(f"{h} h", o) for h, o in zip(HOURS, outputs)]).set_index("case")

The Aitken peak erodes as coagulation feeds the accumulation mode, while condensing H2SO4
grows the surviving particles — the classic modal picture of aerosol aging.

## 3. What each process contributes

Six hours with different processes enabled, everything else identical.

This uses a *clean* background (1% of the loading above) with more H2SO4, because
nucleation is strongly suppressed by a condensation sink: under the polluted scenario,
enabling it changes total number by well under a percent — the fresh clusters are
scavenged as fast as they form. In clean air the same switch changes the answer by two
orders of magnitude, which is exactly the regime dependence a model intercomparison needs
to probe.

In [ ]:
CASES = [
    ("coagulation only", ambrs.AerosolProcesses(coagulation=True)),
    ("+ condensation", ambrs.AerosolProcesses(coagulation=True, condensation=True,
                                              gas_phase_chemistry=True)),
    ("+ nucleation", ambrs.AerosolProcesses(coagulation=True, condensation=True,
                                            nucleation=True, gas_phase_chemistry=True)),
]

# a clean background (1% of the polluted loading) with 0.25 ppb of H2SO4
clean_scenario = ambrs.Scenario(
    aerosols=(so4, soa, ncl, pom, bc), gases=(so2, h2so4),
    size=aerosol.AerosolModalSizeState(modes=(
        mode("accumulation",   [so4, soa, ncl], [0.3, 0.3, 0.4], 3e7,  0.11e-6, 1.8),
        mode("aitken",         [so4, soa, ncl], [0.3, 0.3, 0.4], 3e8,  0.026e-6, 1.6),
        mode("coarse",         [so4, soa, ncl], [0.3, 0.3, 0.4], 1e3,  2.0e-6, 1.8),
        mode("primary carbon", [pom, bc],       [0.5, 0.5],      6e7,  0.05e-6, 1.6),
    )),
    gas_concs=(1e-4, 1e-9),   # SO2, H2SO4 [kg/kg]
    flux=0.0, relative_humidity=0.9, temperature=273.0,
    pressure=1.0e5, height=500.0,
)

nstep = int(6 * 3600 / DT)
process_outputs = []
for label, processes in CASES:
    case_model = ambrs.mam4_jax.AerosolModel(processes)
    process_outputs.append(case_model.run(
        case_model.create_input(clean_scenario, dt=DT, nstep=nstep), label))

fig, ax = plt.subplots()
for colour, (label, _), output in zip(PALETTE, CASES, process_outputs):
    ax.plot(DIAMETERS * 1e9, size_distribution(output), lw=2, color=colour, label=label)
style_axes(ax, "Contribution of each process after 6 h (clean background)")
ax.set_yscale("log")   # nucleation changes number by orders of magnitude
ax.set_ylim(bottom=1e4)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

pd.DataFrame([summarise(label, o)
              for (label, _), o in zip(CASES, process_outputs)]).set_index("case")

## 4. Watch the modes themselves

A modal model's natural diagnostics are per-mode: number concentration and geometric mean
diameter. Both are read straight off the model state (`q`'s number tracers and `dgncur_a`),
here reconstructed from the output populations at a few times.

In [ ]:
# per-mode number and diameter directly from MAM4's state, via a short re-run
import mam4_jax  # noqa: F401  (sets float64; safe, already imported via ambrs)
from ambrs.mam4_jax import NUMPTR_AMODE, NMODES, _air_density
import jax.numpy as jnp

sample_hours = np.array([0, 2, 4, 8, 12, 18, 24])
mode_number = np.zeros((len(sample_hours), NMODES))
mode_diam = np.zeros((len(sample_hours), NMODES))

rho_air = _air_density(scenario.temperature, scenario.pressure)
for i, h in enumerate(sample_hours):
    inp = model.create_input(scenario, dt=DT, nstep=int(h * 3600 / DT) or 1)
    step = model.step_function(
        {"mdo_gasaerexch": inp.mdo_gasaerexch, "mdo_rename": inp.mdo_rename,
         "mdo_newnuc": inp.mdo_newnuc, "mdo_coag": inp.mdo_coag},
        inp.gaschem_rate)
    state = {
        "q": jnp.asarray(inp.q), "qqcw": jnp.asarray(inp.qqcw),
        "dgncur_a": jnp.asarray(inp.dgncur_a),
        "dgncur_awet": jnp.asarray(inp.dgncur_awet),
        "qaerwat": jnp.asarray(inp.qaerwat), "wetdens": jnp.asarray(inp.wetdens),
        "t": jnp.full((1, 1), inp.temp), "pmid": jnp.full((1, 1), inp.press),
        "cldn": jnp.zeros((1, 1)), "zmid": jnp.full((1, 1), inp.zmid),
        "pblh": jnp.full((1, 1), inp.pblh), "relhum": jnp.full((1, 1), inp.rh),
        "deltat": jnp.asarray(float(inp.dt)),
    }
    for _ in range(inp.nstep):
        state = step(state)
    q = np.asarray(state["q"]).reshape(-1)[..., :]
    mode_number[i] = [float(np.asarray(state["q"]).reshape(-1)[p]) * rho_air
                      for p in NUMPTR_AMODE]
    mode_diam[i] = np.asarray(state["dgncur_a"]).reshape(-1)

MODE_NAMES = ["accumulation", "aitken", "coarse", "primary carbon"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
for m in range(NMODES):
    ax1.plot(sample_hours, mode_number[:, m], lw=2, color=PALETTE[m],
             label=MODE_NAMES[m])
    ax2.plot(sample_hours, mode_diam[:, m] * 1e9, lw=2, color=PALETTE[m])
ax1.set_yscale("log"); ax1.set_xlabel("elapsed [h]")
ax1.set_ylabel("mode number [# m$^{-3}$]"); ax1.set_title("Per-mode number")
ax1.legend(frameon=False, fontsize=9)
ax2.set_yscale("log"); ax2.set_xlabel("elapsed [h]")
ax2.set_ylabel("geometric mean diameter [nm]"); ax2.set_title("Per-mode diameter")
plt.tight_layout()
plt.show()

## 5. Comparing against the executable MAM4

The executable MAM4 is the same physics compiled from Fortran; AMBRS drives it through
`PoolRunner` with a namelist, and `ambrs.mam4.retrieve_model_state` builds the same kind of
`Output`:

```python
runner = ambrs.PoolRunner(mam4_model, executable="mam4", root="mam4_runs")
runner.run(mam4_model.create_inputs(ensemble, dt=DT, nstep=nstep))
output = ambrs.mam4.retrieve_model_state("1", scenario, timestep=nstep,
                                         ensemble_output_dir="mam4_runs")
```

Because every model returns the same `ambrs.analysis.Output`, `size_distribution(output)`
above works unchanged on either — and on TOMAS-JAX's sectional output too — so JAX-vs-Fortran
MAM4 parity checks, or modal-vs-sectional comparisons, are one plotting cell away.
`ambrs.analysis.kl_divergence` and `nmae` quantify the differences.